# Averis X Monash Hackathon
**Team dareDEVils**

1. Classification Pipeline testing
2. Comparison & Analytics Strategy testing

Install and Import all the required Libraries and Modules

In [4]:
! pip install pandas
! pip install spacy
! pip install sentence-transformers
! pip install numpy
! pip install scipy

In [ ]:
# for data loading
import pandas as pd
import os
import glob
import json

# for classification
import spacy

# for comparison
import sentence_transformers as st
import numpy as np
import scipy as sp

The data is to be retrieved either directly from data_v2 folder or using a uvicorn FastAPI server with endpoints:

**The endpoints**
 
| Method | Path | Returns |
|---|---|---|
| GET | `/health` | `{"status","emails","scoring_available"}` |
| GET | `/emails` | list of all 520 email records |
| GET | `/emails/{email_id}` | one record, e.g. `/emails/email_004` |
| GET | `/attachments/{path}` | the raw file bytes |
| GET | `/sample_submission` | the exact output shape, all 520 keys |
| POST | `/submit` | scoreboard JSON |
| GET | `/ground_truth` | 404 unless `REVEAL_GT=1` — judges only |
 
`{path}` is the attachment string **minus** the `attachments/` prefix already in
the URL, so `attachments/email_004_SI.txt` → `GET /attachments/email_004_SI.txt`.
Just concatenate: `base_url + "/" + att_string` gives the right URL either way.

The expected input is an email for the given dataset which haas 5 fields and is a json of the format:
```bash
{
  "email_id": "email_XXX",
  "from": "abc1234@pqrs.xxx",
  "subject": "XXXX YYYY ZZZZ",
  "body": "lorem ipsum ........",
  "attachments": ["attachments/email_XXX_SI.yyy", "attachments/email_XXX_BL.yyy"]
}
```

Load the input data directly from data_v2/inbox using pandas

In [7]:
# List out all the filenames of the email json files in inbox directory
inbox_dir = "data_v2/inbox"
mail_files = os.path.join(inbox_dir, "*.json") # all mail as json files
mail_files_list = glob.glob(mail_files) # list of all json files

# for each mail file, read and add the mail information to dataframe
mail_data = []
for mail_file in mail_files_list:
    with open(mail_file, "r") as mfile:
        mail_data.append(pd.json_normalize(json.loads(mfile.read())))

# convert the extracted json fields data to dataframe
mail_df = pd.concat(mail_data)

## Classification

**Current Plan:**

1. Spacy similarity matching (Primary Comparison)
2. LLM Call (Confidence Score based Fallback)

In [8]:
"""
Classification Pipeline
"""

# mail has to be classified into one of these categories
# {label: explanation}
mail_categories = {
    "BL_COMPARISON": "Comparison requested for the Bill of Landing (BL) and Shipping Instruction (SI)",
    "SI_REQUEST": "Request for new Shipping Information (SI)",
    "INVOICE_QUERY": "Query about Invoice",
    "GENERAL": "General Messages",
    "SPAM": "Spam mails (unwanted messages)"
}

# Comparison

**Current Plan:**
Multi-Stage analysis and escalation as required
1. REGEX + Levenshtein distance
2. Vector Embeddings for grouping
3. Further Extraction and Numerical Units, Product Features and Dimensional Comparison
4. LLM Call (Confidence Score based Fallback)
5. Human in the Loop

In [ ]:
"""
Helper Functions
"""

class SemanticMatchingEngine:
    """
    Class to calculate the similarity scores between words
    and word lists
    """
    def __init__(self):
        """
        Load and cache the encoder to be used
        """
        # BERT based Mini Language Model to get sentence embeddings
        # essentially used as an encoder
        self.model = st.SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    def cosine_similarity(self, words_1, words_2):
        """
        Takes 2 lists of words, converts them to embeddings, and finally does
        cosine similarity scoring on the embeddings to get the similarity scores
        for all the possible word combinations across the 2 lists.
                                     a   b   c
        [[x],                    x   s1  s2  s3
         [y],  X   [a, b, c]  =  y   s4  s5  s6
         [z]]                    z   s7  s8  s9

        The matrix multiplication gives a NxN vector with all the scores s1, s2, ....si
        """
        # convert words to embeddings
        embeddings_1 = self.model.encode(words_1, normalize_embeddings=True)
        embeddings_2 = self.model.encode(words_2, normalize_embeddings=True)

        # vector cross product to get the similarity scores for all the combinations of words
        cosine_similarity_scores = embeddings_1 @ embeddings_2.T
        return cosine_similarity_scores

    def get_semantically_similar_words(self, words_1, words_2, threshold=0.0):
        """
        get the most similar words across 2 lists of words

        [a, b, c] [y, z, x] -> [[a, x], [b, y], [c, z]]
        gives the best match across list and the best score
        for threshold comparison
        """
        scores = self.cosine_similarity(words_1, words_2)  # N1 x N2

        # linear sum assignment optimizes by picking the smalles combination so
         # negate score to maximize score matching similarity
        row_idx, col_idx = sp.optimize.linear_sum_assignment(-scores)

        matches = []
        for i, j in zip(row_idx, col_idx):
            score = round(float(scores[i, j]), 3)
            if score >= threshold:
                matches.append((words_1[i], words_2[j], score))

        return matches

    def words_clustering(self, words, tolerance=0.3):
        """
        Takes a list of words and clusters the words into a group
        with similar words from the same list
        (Aggregator similar to k-means clustering)

        [a,b,c,p,q,x,z] -> [[a,b,c], [p,q], [x,z]]

        returns the list of clusters with each cluster having closely associated words
        *Note: pass normalized input words for better accuracy
        """
        # convert words to embeddings using the encoder transformer (based off of BERT)
        embeddings = self.model.encode(words, normalize_embeddings=True)

        # clustering model
        clustering = st.AgglomerativeClustering(
            n_clusters=None,
            distance_threshold=tolerance,
            metric="cosine",
            linkage="average"
        )

        # cluster labels
        labels = clustering.fit_predict(embeddings)

        # group words by cluster label
        clusters = {}
        for word, label in zip(words, labels):
            clusters.setdefault(label, []).append(word)

        return clusters

In [ ]:
"""
Comparison
"""

''